In [30]:
import numpy as np
import librosa
import soundfile as sf
from scipy.signal import lfilter

In [31]:
# Load audio
y, sr = librosa.load("../raw_audio/dummy.wav", sr=None)

print(f'Processing an audio file with sample rate: {sr}')

Processing an audio file with sample rate: 44100


In [34]:
# LPC settings
frame_length = int(0.03 * sr)   # 30 ms
hop_length = frame_length // 2  # 50% overlap
order = 16                      # no of past frames in regression model

# Pad audio for framing
y = np.pad(y, (0, frame_length), mode="constant")

# Create overlapping frames
frames = librosa.util.frame(
    y,
    frame_length=frame_length,
    hop_length=hop_length
)

print(f'Extracted array of {frames.shape[1]} frames, each with dim {frames.shape[0]}')

Extracted array of 201 frames, each with dim 1323


In [35]:
# Hann window
window = np.hanning(frame_length)

# Output buffers
output = np.zeros(len(y))
residual_output = np.zeros(len(y))
norm = np.zeros(len(y))

In [36]:
# Process each frame
for i in range(frames.shape[1]):

    # Apply hann window
    frame = frames[:, i] * window

    # LPC coefficient calculation (filter) for frame
    a = librosa.lpc(frame, order=order)

    # Compute residual (inverse filtering)
    residual = lfilter(a, [1.0], frame)

    # Reconstruct signal from residual
    reconstructed = lfilter([1.0], a, residual)

    # Starting at hop length, this was encoded in frame generation
    start = i * hop_length

    output[start:start + frame_length] += reconstructed * window
    residual_output[start:start + frame_length] += residual * window

    # Window normalization
    norm[start:start + frame_length] += window**2

# Normalize overlap regions
output /= (norm + 1e-8)
residual_output /= (norm + 1e-8)

# Save outputs
sf.write("reconstructed.wav", output, sr)
sf.write("residual.wav", residual_output, sr)